In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir("../")

In [3]:
from ease_recommender import *
from npmi_recommender import *

import pickle as p

In [4]:
print("loading cache data...")
D = p.load(open("cached_data/movie_lens_preprocessed.p", "rb"))

row = D["userId"]
col = D["movieId"]
data = D["rating"]

movies = D["movies"]
movies = movies[(movies.groupby("title").num_votes.transform("max") == movies.num_votes)]

print("done")

loading cache data...
done


In [16]:
def find_match_using_terms(terms, movies=movies, case_insensitive=False):
    assert type(terms) in (list, tuple, set)
    
    title = movies.title
    if case_insensitive:
        title = title.str.lower()
    
    matches = True
    for term in terms:
        matches &= title.str.contains(term)
        
    matches = np.where(matches)[0]
    
    if len(matches) > 1:
        raise Exception("Multiple matches found, filter down to a single match", movies.loc[matches, "title"].tolist())
        
    return matches[0]

In [6]:
np.unique(data)

array([ 7,  8,  9, 10])

In [103]:
m = data >= 7
# m = data >= 9

In [104]:
mat = csr_array((data[m].astype(bool), (row[m], col[m]))).astype(np.int64)

In [105]:
X = mat.T @ mat

In [106]:
X.shape

(64548, 64548)

In [107]:
import numpy as np
from scipy import sparse
from scipy.optimize import minimize

class SppmiOptimizerCache:
    """Caches global raw matrix invariants in memory for rapid lookup loops."""
    def __init__(self, X):
        self.X_csc = X.tocsc() if not sparse.isspmatrix_csc(X) else X
        self.rows, self.cols = self.X_csc.shape
        self.N_raw = self.X_csc.sum()
        self.row_sums_raw = np.array(self.X_csc.sum(axis=1)).flatten()
        self.col_sums_raw = np.array(self.X_csc.sum(axis=0)).flatten()


def compute_item_scores(cache, a, alpha, gamma, tau, zero_diag=True):
    """Generates the full 1D similarity profile vector for an item 'a'."""
    rows, cols = cache.rows, cache.cols
    N_smoothed = cache.N_raw + (alpha * rows * cols)
    
    col_a = cache.X_csc[:, a:a+1]
    curr_rows, curr_data = col_a.indices, col_a.data.astype(np.float32)
    
    P_x = (cache.row_sums_raw + (alpha * cols)) / N_smoothed
    P_y_a = (cache.col_sums_raw[a] + (alpha * rows)) / N_smoothed
    
    scores = np.zeros(rows, dtype=np.float32)
    
    # SHORT-CIRCUIT OPTIMIZATION: If tau is disabled but alpha is 0, 
    # all structural zeros are 0.0 anyway. We can skip full dense grid calculations.
    if tau > 0 or alpha == 0:
        P_xy_nz = (curr_data + alpha) / N_smoothed
        scores_nz = P_xy_nz / ((P_x[curr_rows] ** gamma) * (P_y_a ** gamma))
        if tau > 0:
            scores_nz *= (curr_data / (curr_data + tau))
        scores[curr_rows] = scores_nz
    else:
        # Full dense field processing (only active if alpha > 0 AND tau == 0)
        P_xy = np.full(rows, alpha / N_smoothed, dtype=np.float32)
        P_xy[curr_rows] = (curr_data + alpha) / N_smoothed
        scores = P_xy / ((P_x ** gamma) * (P_y_a ** gamma))
        
    if zero_diag:
        scores[a] = 0.0
        
    return scores


def optimize_sppmi_parameters(cache, item_groups, init_params=None, optimize_vars=None):
    """
    Optimizes selective SPPMI parameters concurrently using Powell's method.
    Bypasses and neutralizes parameters explicitly set to None.
    """
    default_params = {'alpha': 0.1, 'gamma': 1.0, 'tau': 2.0}
    neutral_defaults = {'alpha': 0.0, 'gamma': 1.0, 'tau': 0.0}

    # Clean input dictionaries
    if init_params is None:
        init_params = default_params.copy()
    else:
        for k, v in default_params.items():
            if k not in init_params:
                init_params[k] = v

    if optimize_vars is None:
        optimize_vars = ['alpha', 'gamma', 'tau']

    # CRITICAL: Validate and substitute None values with neutral defaults
    for var in ['alpha', 'gamma', 'tau']:
        if init_params[var] is None:
#             assert var not in optimize_vars, f"AssertionError: Cannot optimize '{var}' because it is set to None."
            init_params[var] = neutral_defaults[var]

    for i, group in enumerate(item_groups):
        assert len(group) >= 2, f"Group {i} has fewer than 2 items. Cannot compute mutual ranks."

    unique_items = list(set(idx for group in item_groups for idx in group))
    x0 = [init_params[var] for var in optimize_vars]

    def objective(x_vec):
        current_params = init_params.copy()
        for var_name, val in zip(optimize_vars, x_vec):
            current_params[var_name] = val
            
        alpha, gamma, tau = current_params['alpha'], current_params['gamma'], current_params['tau']
        
        # Hard boundary protection penalties to block line searches into illegal spaces
        # (Allows alpha to be exactly 0.0 now if it was intentionally passed as None)
        if alpha < 0.0 or gamma <= 0.0 or tau < 0.0:
            return 1e9
            
        current_scores = {idx: compute_item_scores(cache, idx, alpha, gamma, tau) for idx in unique_items}
        group_scores = []
        
        for group in item_groups:
            total_pairwise_rank = 0.0
            k = len(group)
            
            for u in group:
                sort_indices = np.argsort(-current_scores[u])
                ranks = np.empty_like(sort_indices)
                ranks[sort_indices] = np.arange(len(sort_indices))
                
                for v in group:
                    if u == v:
                        continue
                    total_pairwise_rank += ranks[v]
                    
            group_scores.append(total_pairwise_rank / (k * (k - 1)))
            
        return np.mean(group_scores)

    # Execute multidimensional optimization if there are active variables to track
    if len(optimize_vars) > 0:
        res = minimize(objective, x0, method='Powell')
        final_values = res.x if len(optimize_vars) > 1 else [res.x]
        
        final_params = init_params.copy()
        for var_name, val in zip(optimize_vars, final_values):
            final_params[var_name] = float(val)
        return res.fun, final_params
    else:
        # Short circuit optimization if all variables are locked/None
        return objective(init_params), init_params

In [108]:
cache = SppmiOptimizerCache(X)

In [109]:
item_group_a = [
    find_match_using_terms(["Sense and Sensibility", "1995"]),
    find_match_using_terms(["Pride and Prejudice", "1995"])
]

In [110]:
item_group_b = [
    find_match_using_terms(["Prince of Egypt"]),
    find_match_using_terms(["Aladdin", "1992"])
]

In [111]:
nested_items = [
    item_group_a,
#     item_group_b,
]

In [112]:
# Optimize everything at once
best_score, best_params = optimize_sppmi_parameters(
    cache, 
    item_groups=nested_items,
    init_params={'alpha': None, 'gamma': None, 'tau': None}, # Initialize them to neutral defaults
    optimize_vars=['tau', 'gamma', 'alpha'] # <- Tells the engine to tune all three
)

print("Optimized Parameters:", best_params)
print("Best Achieved Mean Rank:", best_score)

Optimized Parameters: {'alpha': 0.64956364441953, 'gamma': 1.148966648294391, 'tau': 222.76751629028942}
Best Achieved Mean Rank: 31.5


In [113]:
# Optimize everything at once
best_score, best_params = optimize_sppmi_parameters(
    cache, 
    item_groups=nested_items,
    init_params={'alpha': None, 'gamma': None, 'tau': None}, # Initialize them to neutral defaults
#     optimize_vars=['alpha'] # <- Tells the engine to tune all three
#     optimize_vars=['tau'] # <- Tells the engine to tune all three
#     optimize_vars=['gamma'] # <- Tells the engine to tune all three
#     optimize_vars=['alpha', 'tau'] # <- Tells the engine to tune all three
#     optimize_vars=['tau', 'alpha'] # <- Tells the engine to tune all three
#     optimize_vars=['alpha', 'gamma'] # <- Tells the engine to tune all three
#     optimize_vars=['gamma', 'alpha'] # <- Tells the engine to tune all three
#     optimize_vars=['tau', 'gamma'] # <- Tells the engine to tune all three
#     optimize_vars=['gamma', 'tau'] # <- Tells the engine to tune all three
 
#     optimize_vars=['gamma', 'tau', 'alpha'] # <- Tells the engine to tune all three
#     optimize_vars=['gamma', 'alpha', 'tau'] # <- Tells the engine to tune all three
#     optimize_vars=['alpha', 'gamma', 'tau'] # <- Tells the engine to tune all three
#     optimize_vars=['alpha', 'tau', 'gamma'] # <- Tells the engine to tune all three
#     optimize_vars=['tau', 'alpha', 'gamma'] # <- Tells the engine to tune all three
    optimize_vars=['tau', 'gamma', 'alpha'] # <- Tells the engine to tune all three
)

print("Optimized Parameters:", best_params)
print("Best Achieved Mean Rank:", best_score)

Optimized Parameters: {'alpha': 0.64956364441953, 'gamma': 1.148966648294391, 'tau': 222.76751629028942}
Best Achieved Mean Rank: 31.5


In [114]:
a, b = item_group_a

In [115]:
# lambda_ = optimize_lambda_using_a_to_b_matching(mat, a, b, fast_approximation=False)
lambda_ = optimize_lambda_using_a_to_b_matching(mat, a, b, fast_approximation=True)

lambda_: 1000000
error: 182.999999
lambda_: 100000.0
error: 42.99999
lambda_: 10000.0
error: 11.9999
lambda_: 1000.0
error: 9.999
lambda_: 100.0
error: 13.99


In [116]:
# check error for EASE with optimized lambda_
a_to_b_error_metric(mat, a, b, lambda_, lambda_penalty=False)

lambda_: 1000
error: 9


9

In [117]:
top_k = 20

In [118]:
# using EASE

similarity_scores = calculate_ease_for_item_cg(mat, a, lambda_)

top_k_matches = movies.loc[np.argsort(-similarity_scores)[:top_k].tolist()]

top_k_matches

,title,genres,imdbId,tmdbId,avg_rating,num_votes
movieId,,,,,,
819,Emma (1996),Comedy|Drama|Romance,116191,3573.0,7.485110,6486
27,Persuasion (1995),Drama|Romance,114117,17015.0,8.076637,2719
7382,Pride and Prejudice (1995),Drama|Romance,112130,164721.0,7.989470,2607
510,"Remains of the Day, The (1993)",Drama|Romance,107943,1245.0,7.783682,8651
605,Jane Eyre (1996),Drama|Romance,116684,47333.0,7.271146,1807
57,"Postman, The (Postino, Il) (1994)",Comedy|Drama|Romance,110877,11010.0,7.927828,10200
10352,Pride & Prejudice (2005),Drama|Romance,414387,4348.0,7.703508,6645
492,Much Ado About Nothing (1993),Comedy|Romance,107616,11971.0,7.734885,11266
258,Little Women (1994),Drama,110367,9587.0,7.199093,7447


In [119]:
# using EASE

similarity_scores = calculate_ease_for_item_cg(mat, b, lambda_)

top_k_matches = movies.loc[np.argsort(-similarity_scores)[:top_k].tolist()]

top_k_matches

,title,genres,imdbId,tmdbId,avg_rating,num_votes
movieId,,,,,,
14199,Persuasion (2007),Drama|Romance,844330,13949.0,7.719715,330
17597,North & South (2004),Drama|Romance,417349,147269.0,8.057732,400
10352,Pride & Prejudice (2005),Drama|Romance,414387,4348.0,7.703508,6645
13158,"Young Victoria, The (2009)",Drama|Romance,962736,18320.0,7.472767,679
27,Persuasion (1995),Drama|Romance,114117,17015.0,8.076637,2719
15923,Jane Eyre (2011),Drama|Romance,1229822,38684.0,7.451439,824
16728,Northanger Abbey (2007),Drama|Romance,844794,18093.0,7.429658,201
11401,Becoming Jane (2007),Drama|Romance,416508,2977.0,7.129006,837
2984,Mansfield Park (1999),Comedy|Drama|Romance,178737,10399.0,7.523058,1191


In [120]:
# average the two rankings

similarity_scores_a = calculate_ease_for_item_cg(mat, a, lambda_)
similarity_ranking_a = np.full(len(similarity_scores_a), -1.0)
similarity_ranking_a[np.argsort(-similarity_scores_a)] = 1.0 - ((1.0 + np.arange(len(similarity_ranking_a)))/len(similarity_ranking_a))

similarity_scores_b = calculate_ease_for_item_cg(mat, b, lambda_)
similarity_ranking_b = np.full(len(similarity_scores_b), -1.0)
similarity_ranking_b[np.argsort(-similarity_scores_b)] = 1.0 - ((1.0 + np.arange(len(similarity_ranking_b)))/len(similarity_ranking_b))

similarity_scores = similarity_ranking_a * similarity_ranking_b
# similarity_scores = similarity_scores_a * similarity_scores_b

top_k_matches = movies.loc[np.argsort(-similarity_scores)[:top_k].tolist()]

for m in movies.loc[np.argsort(-similarity_scores)[:top_k], "title"]:
    print(m)

Persuasion (1995)
Pride & Prejudice (2005)
Emma (1996)
Mansfield Park (1999)
Becoming Jane (2007)
Persuasion (2007)
North & South (2004)
Jane Eyre (2011)
Importance of Being Earnest, The (2002)
Northanger Abbey (2007)
Jane Eyre (1996)
Duchess, The (2008)
Roman Holiday (1953)
Far from the Madding Crowd (2015)
Bridget Jones's Diary (2001)
The Queen (2006)
Fiddler on the Roof (1971)
Pride and Prejudice (1940)
Phantom of the Opera, The (2004)
Room with a View, A (1986)


In [121]:
a, b = item_group_a

In [122]:
top_k = 15

similarity_scores = compute_item_scores(
    cache, 
    a=a,
    **best_params
)

top_k_matches = movies.loc[np.argsort(-similarity_scores)[:top_k].tolist()]

top_k_matches

,title,genres,imdbId,tmdbId,avg_rating,num_votes
movieId,,,,,,
605,Jane Eyre (1996),Drama|Romance,116684,47333.0,7.271146,1807
27,Persuasion (1995),Drama|Romance,114117,17015.0,8.076637,2719
819,Emma (1996),Comedy|Drama|Romance,116191,3573.0,7.485110,6486
138,Up Close and Personal (1996),Drama|Romance,118055,9302.0,6.615276,2295
34,Carrington (1995),Drama|Romance,112637,47018.0,6.987390,837
219,Circle of Friends (1995),Drama|Romance,112679,22625.0,7.057626,3441
57,"Postman, The (Postino, Il) (1994)",Comedy|Drama|Romance,110877,11010.0,7.927828,10200
447,Widows' Peak (1994),Drama,111712,25440.0,7.119125,646
258,Little Women (1994),Drama,110367,9587.0,7.199093,7447


In [123]:
top_k = 15

similarity_scores = compute_item_scores(
    cache, 
    a=b,
    **best_params
)

top_k_matches = movies.loc[np.argsort(-similarity_scores)[:top_k].tolist()]

top_k_matches

,title,genres,imdbId,tmdbId,avg_rating,num_votes
movieId,,,,,,
14199,Persuasion (2007),Drama|Romance,844330,13949.0,7.719715,330
17597,North & South (2004),Drama|Romance,417349,147269.0,8.057732,400
16728,Northanger Abbey (2007),Drama|Romance,844794,18093.0,7.429658,201
11401,Becoming Jane (2007),Drama|Romance,416508,2977.0,7.129006,837
10352,Pride & Prejudice (2005),Drama|Romance,414387,4348.0,7.703508,6645
13158,"Young Victoria, The (2009)",Drama|Romance,962736,18320.0,7.472767,679
27,Persuasion (1995),Drama|Romance,114117,17015.0,8.076637,2719
15923,Jane Eyre (2011),Drama|Romance,1229822,38684.0,7.451439,824
2984,Mansfield Park (1999),Comedy|Drama|Romance,178737,10399.0,7.523058,1191


In [126]:
# average the two rankings

similarity_scores_a = compute_item_scores(
    cache, 
    a=a,
    **best_params
)
similarity_ranking_a = np.full(len(similarity_scores_a), -1.0)
similarity_ranking_a[np.argsort(-similarity_scores_a)] = 1.0 - ((1.0 + np.arange(len(similarity_ranking_a)))/len(similarity_ranking_a))

similarity_scores_b = compute_item_scores(
    cache, 
    a=b,
    **best_params
)
similarity_ranking_b = np.full(len(similarity_scores_b), -1.0)
similarity_ranking_b[np.argsort(-similarity_scores_b)] = 1.0 - ((1.0 + np.arange(len(similarity_ranking_b)))/len(similarity_ranking_b))

# similarity_scores = similarity_ranking_a * similarity_ranking_b
similarity_scores = similarity_scores_a * similarity_scores_b

top_k_matches = movies.loc[np.argsort(-similarity_scores)[:top_k].tolist()]

for m in movies.loc[np.argsort(-similarity_scores)[:top_k], "title"]:
    print(m)

Persuasion (1995)
Emma (1996)
Persuasion (2007)
Jane Eyre (1996)
North & South (2004)
Mansfield Park (1999)
Becoming Jane (2007)
Northanger Abbey (2007)
Little Women (1994)
Pride & Prejudice (2005)
Young Victoria, The (2009)
Much Ado About Nothing (1993)
Room with a View, A (1986)
Sabrina (1995)
Remains of the Day, The (1993)


In [95]:
# a, b = item_group_a

# scores_a = scores_b = similarity_scores

# (np.argsort(-scores_a).tolist().index(b) + np.argsort(-scores_b).tolist().index(a))/2